# Chapter 10 — Neural Networks Learn Structure: Convolutional Layers

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning, 2019)
**Chapter 10:** Reusing weights in multiple places: the convolutional layer

## Isi Chapter 10

1. Reuse bobot: mengapa struktur membantu melawan overfitting
2. Convolutional layer: banyak kernel kecil yang dipakai di setiap posisi
3. Implementasi NumPy: mengambil sub-region dan meratakannya
4. Mengecek bahwa implementasi itu sama dengan konvolusi biasa
5. Menghitung jumlah parameter
6. Training MNIST dengan convolutional layer

## Catatan data

Bagian 6 memakai dataset MNIST dari `keras.datasets`, seperti Chapter 8 dan 9.
Jalankan bagian itu di Google Colab. Bagian 1–5 tidak memerlukan MNIST.

## 1. Reuse bobot

Overfitting terjadi ketika jumlah parameter terlalu banyak dibanding jumlah contoh
latih. Regularization (Chapter 8) adalah salah satu cara melawannya. Cara lain adalah
memakai **struktur**: sebuah bobot dipakai ulang di banyak tempat, karena pola yang
sama mungkin muncul di banyak posisi. Dengan begitu, jumlah parameter berkurang, tetapi
kapasitas model untuk mengenali pola tetap ada.

Convolution adalah struktur paling terkenal. Saat dipakai sebagai layer, namanya
convolutional layer.

## 2. Convolutional layer

Alih-alih satu layer linear besar yang terhubung ke setiap input, saya memakai banyak
layer linear kecil (biasanya kurang dari 25 input dan 1 output) yang dipindahkan ke
setiap posisi gambar. Setiap layer kecil ini disebut **kernel**.

Contoh: kernel 3 x 3 membaca area 3 x 3, bergeser satu piksel ke kanan, membaca lagi,
dan seterusnya sampai seluruh gambar dipindai. Pada gambar 8 x 8, hasilnya adalah
matriks 6 x 6 (karena 8 - 3 + 1 = 6). Jika ada 4 kernel, saya mendapat 4 matriks 6 x 6.
Matriks-matriks itu digabung dengan **max pooling**: untuk setiap posisi, nilai
maksimum dari keempat kernel diambil. Hasilnya diteruskan ke layer berikutnya.

Kernel bisa mempelajari satu pola (misalnya garis horizontal) dan mencarinya di seluruh
gambar. Kernel yang tidak menemukan polanya akan mengeluarkan nilai rendah.

## 3. Implementasi NumPy

Saya mulai dari fungsi untuk mengambil sub-region dari sebuah batch gambar. Fungsi ini
mengambil area yang sama dari seluruh batch sekaligus.

In [ ]:
import numpy as np

def get_image_section(layer, row_from, row_to, col_from, col_to):
    section = layer[:, row_from:row_to, col_from:col_to]
    return section.reshape(-1, 1, row_to - row_from, col_to - col_from)

# Contoh: batch 2 gambar 8x8, ambil area 3x3 di pojok kiri atas
batch = np.arange(2 * 8 * 8).reshape(2, 8, 8)
sec = get_image_section(batch, 0, 3, 0, 3)
print('shape hasil :', sec.shape)     # (batch, 1, 3, 3)
print(sec[0, 0])

Untuk mendapatkan semua posisi, saya memanggil fungsi ini di setiap posisi, lalu
menggabungkan hasilnya. Dengan cara ini, convolution berubah menjadi satu perkalian
matriks biasa:

- `flattened_input` berisi setiap sub-region yang sudah diratakan menjadi vektor 9 nilai.
- `kernels` berisi 9 bobot untuk setiap kernel.
- `flattened_input.dot(kernels)` menghitung semua kernel di semua posisi sekaligus.

Saya cek bahwa cara ini sama dengan konvolusi yang ditulis dengan loop biasa.

In [ ]:
def expand_input(layer_0, kernel_rows, kernel_cols):
    # layer_0: (batch, rows, cols) -> (batch * n_positions, kernel_rows * kernel_cols)
    sects = list()
    for row_start in range(layer_0.shape[1] - kernel_rows + 1):
        for col_start in range(layer_0.shape[2] - kernel_cols + 1):
            sect = get_image_section(layer_0,
                                     row_start, row_start + kernel_rows,
                                     col_start, col_start + kernel_cols)
            sects.append(sect)
    expanded_input = np.concatenate(sects, axis=1)
    es = expanded_input.shape
    return expanded_input.reshape(es[0] * es[1], -1), es

def conv_loops(layer_0, kernels_mat, kernel_rows, kernel_cols):
    # Konvolusi dengan loop biasa, untuk pembanding.
    batch, rows, cols = layer_0.shape
    num_kernels = kernels_mat.shape[1]
    out = np.zeros((batch, rows - kernel_rows + 1, cols - kernel_cols + 1, num_kernels))
    for b in range(batch):
        for r in range(rows - kernel_rows + 1):
            for c in range(cols - kernel_cols + 1):
                patch = layer_0[b, r:r+kernel_rows, c:c+kernel_cols].reshape(-1)
                out[b, r, c, :] = patch.dot(kernels_mat)
    return out.reshape(batch, -1)

rng = np.random.default_rng(0)
kernel_rows, kernel_cols, num_kernels = 3, 3, 4
imgs = rng.random((5, 8, 8))
kernels_mat = rng.random((kernel_rows * kernel_cols, num_kernels)) - 0.5

flattened, es = expand_input(imgs, kernel_rows, kernel_cols)
via_dot = flattened.dot(kernels_mat).reshape(es[0], -1)
via_loops = conv_loops(imgs, kernels_mat, kernel_rows, kernel_cols)

print('shape flattened :', flattened.shape)   # (5 * 36, 9)
print('shape hasil dot:', via_dot.shape)     # (5, 6*6*4)
print('sama dengan loop?', np.allclose(via_dot, via_loops))

Hasilnya sama. Jadi `flattened_input.dot(kernels)` adalah konvolusi, hanya dituliskan
sebagai satu perkalian matriks yang lebih cepat.

## 4. Menemukan pola dengan satu kernel

Untuk melihat kernel bekerja, saya membuat gambar 8 x 8 dengan garis horizontal di
tengah, dan kernel 3 x 3 yang bobotnya cocok dengan garis horizontal. Respons kernel
seharusnya paling tinggi di posisi garis itu.

In [ ]:
image = np.zeros((8, 8))
image[4, 1:7] = 1.0                 # garis horizontal di baris 4

horizontal_kernel = np.array([[-1, -1, -1],
                              [ 2,  2,  2],
                              [-1, -1, -1]], dtype=float).reshape(-1, 1)

flat, es = expand_input(image[np.newaxis], 3, 3)
response = flat.dot(horizontal_kernel).reshape(6, 6)

print(np.round(response, 1))
r, c = np.unravel_index(np.argmax(response), response.shape)
print('respons tertinggi di posisi (baris, kolom) =', (r, c))

Respons tertinggi berada di sekitar baris tengah gambar, tempat garis horizontal berada.
Kernel ini tidak perlu tahu di mana garisnya: ia cukup memindai, lalu menandai posisi
yang cocok dengan polanya. Inilah alasan convolution bisa menggeneralisasi.

## 5. Jumlah parameter

Dengan reuse bobot, jumlah parameter berubah. Saya bandingkan dua layer pertama untuk
MNIST (28 x 28 input):

In [ ]:
input_rows, input_cols = 28, 28
kernel_rows, kernel_cols, num_kernels = 3, 3, 16
num_labels = 10

# Layer konvolusi: hanya 9 bobot per kernel, dipakai di semua posisi
conv_params = kernel_rows * kernel_cols * num_kernels
hidden_size = ((input_rows - kernel_rows + 1) * (input_cols - kernel_cols + 1)) * num_kernels

# Layer hidden_size -> output (weights_1_2)
out_params_conv = hidden_size * num_labels

# Layer dense 784 -> 100 untuk perbandingan (dari chapter 9)
dense_params = 784 * 100

print('posisi kernel       :', (input_rows - kernel_rows + 1) * (input_cols - kernel_cols + 1))
print('hidden_size         :', hidden_size)
print('parameter konvolusi :', conv_params)
print('parameter output    :', out_params_conv)
print('parameter dense 784x100 (chapter 9):', dense_params)

Layer konvolusi hanya memakai 144 bobot untuk 16 kernel, jauh lebih sedikit daripada
layer dense 784 x 100. Namun, output-nya tetap berukuran 10.816 node (676 posisi x 16
kernel), karena setiap kernel menghasilkan nilai di setiap posisi. Ukuran 26 x 26 untuk
posisi ini diperoleh dari 28 - 3 + 1 = 26. Ini menunjukkan bahwa reuse bobot membuat
layer pertama jauh lebih kecil, sedangkan layer berikutnya tetap bisa besar.

## 6. Training MNIST dengan convolutional layer

Bagian ini mengganti layer pertama dengan convolutional layer. Layer kedua dan
output tetap memakai tanh dan softmax seperti Chapter 9. Jalankan di Colab.

In [ ]:
import numpy as np, sys
from keras.datasets import mnist

np.random.seed(1)
(x_train, y_train), (x_test, y_test) = mnist.load_data()

images = x_train[0:1000].reshape(1000, 28 * 28) / 255
labels_raw = y_train[0:1000]
labels = np.zeros((len(labels_raw), 10))
for i, l in enumerate(labels_raw):
    labels[i][l] = 1

test_images = x_test.reshape(len(x_test), 28 * 28) / 255
test_labels = np.zeros((len(y_test), 10))
for i, l in enumerate(y_test):
    test_labels[i][l] = 1

def tanh(x):
    return np.tanh(x)

def tanh2deriv(output):
    return 1 - (output ** 2)

def softmax(x):
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

alpha, iterations = (2, 300)
pixels_per_image, num_labels = (784, 10)
batch_size = 128
input_rows, input_cols = 28, 28
kernel_rows, kernel_cols = 3, 3
num_kernels = 16
hidden_size = ((input_rows - kernel_rows + 1) * (input_cols - kernel_cols + 1)) * num_kernels

kernels = 0.02 * np.random.random((kernel_rows * kernel_cols, num_kernels)) - 0.01
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

def get_image_section(layer, row_from, row_to, col_from, col_to):
    section = layer[:, row_from:row_to, col_from:col_to]
    return section.reshape(-1, 1, row_to - row_from, col_to - col_from)

def expand(layer_0):
    sects = list()
    for row_start in range(layer_0.shape[1] - kernel_rows + 1):
        for col_start in range(layer_0.shape[2] - kernel_cols + 1):
            sects.append(get_image_section(layer_0,
                                           row_start, row_start + kernel_rows,
                                           col_start, col_start + kernel_cols))
    expanded_input = np.concatenate(sects, axis=1)
    es = expanded_input.shape
    return expanded_input.reshape(es[0] * es[1], -1), es

for j in range(iterations):
    correct_cnt = 0
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end = ((i * batch_size), ((i + 1) * batch_size))

        layer_0 = images[batch_start:batch_end].reshape(-1, 28, 28)
        flattened_input, es = expand(layer_0)
        kernel_output = flattened_input.dot(kernels)
        layer_1 = tanh(kernel_output.reshape(es[0], -1))
        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = softmax(np.dot(layer_1, weights_1_2))

        for k in range(batch_size):
            correct_cnt += int(np.argmax(layer_2[k:k+1]) ==
                               np.argmax(labels[batch_start+k:batch_start+k+1]))

        layer_2_delta = (labels[batch_start:batch_end] - layer_2) / (batch_size * layer_2.shape[0])
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * tanh2deriv(layer_1)
        layer_1_delta *= dropout_mask

        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        l1d_reshape = layer_1_delta.reshape(kernel_output.shape)
        kernels -= alpha * flattened_input.T.dot(l1d_reshape)

    # Evaluasi test setiap 10 iterasi, dan diproses per 1.000 gambar agar lebih cepat.
    if j % 10 == 0 or j == iterations - 1:
        test_correct_cnt = 0
        for t_start in range(0, len(test_images), 1000):
            t_end = min(t_start + 1000, len(test_images))
            layer_0 = test_images[t_start:t_end].reshape(-1, 28, 28)
            flattened_input, es = expand(layer_0)
            kernel_output = flattened_input.dot(kernels)
            layer_1 = tanh(kernel_output.reshape(es[0], -1))
            layer_2 = np.dot(layer_1, weights_1_2)
            test_correct_cnt += int(np.sum(np.argmax(layer_2, axis=1) ==
                                           np.argmax(test_labels[t_start:t_end], axis=1)))
        print('I:', j,
              '| Test-Acc:', round(test_correct_cnt / float(len(test_images)), 4),
              '| Train-Acc:', round(correct_cnt / float(len(images)), 3))

Perbandingan yang saya harapkan: convolutional layer memberi peningkatan akurasi uji
dibanding jaringan Chapter 9 dengan layer dense, dan jumlah parameter di layer pertama
jauh lebih kecil. Angka pastinya belum saya verifikasi, karena dataset MNIST tidak
tersedia di sandbox. Setelah dijalankan di Colab, bandingkan dengan Chapter 9.

Convolutional layer juga biasanya ditumpuk. Setiap layer konvolusi memperlakukan output
layer sebelumnya sebagai gambar. Penumpukan ini adalah salah satu alasan jaringan dalam
(deep) bisa dilatih dengan baik.

## Key Ideas — Chapter 10

- Reuse bobot adalah bentuk struktur yang mengurangi rasio parameter terhadap data, sehingga mengurangi overfitting.
- Kernel adalah layer linear kecil yang dipindahkan ke setiap posisi gambar.
- Convolution bisa ditulis sebagai `flattened_input.dot(kernels)` setelah setiap sub-region diratakan.
- Max pooling mengambil nilai maksimum dari respons semua kernel di setiap posisi.
- Layer konvolusi hanya memiliki sedikit parameter, tetapi outputnya tetap besar karena kernel dipakai di banyak posisi.

## Kesimpulan

Chapter 10 memperkenalkan convolutional layer sebagai cara mereuse bobot di banyak
posisi gambar. Implementasinya menjadi perkalian matriks biasa setelah setiap sub-region
diratakan. Keuntungan utamanya: parameter lebih sedikit dan kemampuan mengenali pola di
mana saja di gambar.